In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer as vectorizer
import pyarrow.parquet as pq
import seaborn as sns
from sklearn.pipeline import Pipeline

In [2]:
data_1 = pq.read_table('/content/data/train-00000-of-00004.parquet')
data_1 = data_1.to_pandas()
data_2 = pq.read_table('/content/data/train-00001-of-00004.parquet')
data_2 = data_2.to_pandas()
data_3 = pq.read_table('/content/data/train-00002-of-00004.parquet')
data_3 = data_3.to_pandas()
data_4 = pq.read_table('/content/data/train-00003-of-00004.parquet')
data_4 = data_4.to_pandas()

In [3]:
df_list = [data_1, data_2, data_3, data_4]
data = pd.concat(df_list, axis=0, ignore_index=True)

In [4]:
print(data.shape,"\n")
print(data.describe(),"\n")
print(data.info(),"\n")

(1175432, 13) 

       is_dangerous  prompt_harmful  prompt_adversarial  response_harmful  \
count  1.175432e+06       1175432.0           1175432.0          259380.0   
mean   6.016222e-01        0.409204            0.273374          0.307568   
std    4.895642e-01        0.491687            0.445692          0.461488   
min    0.000000e+00             0.0                 0.0               0.0   
25%    0.000000e+00             0.0                 0.0               0.0   
50%    1.000000e+00             0.0                 0.0               0.0   
75%    1.000000e+00             1.0                 1.0               1.0   
max    1.000000e+00             1.0                 1.0               1.0   

       response_refusal  
count           39659.0  
mean           0.492776  
std            0.499954  
min                 0.0  
25%                 0.0  
50%                 0.0  
75%                 1.0  
max                 1.0   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1175

In [5]:
data = data[data['language'] == 'en']

In [6]:
x = data['prompt']

In [7]:
y = pd.DataFrame(
(data['prompt_harmful'] == 1) |
(data['prompt_adversarial'] == 1)
).astype(int)
y

,0
0,1
1,1
2,1
3,1
4,1
...,...
1175427,1
1175428,0
1175429,0
1175430,0


In [11]:
print(x.shape,'\n',y.shape)

(990540,) 
 (990540, 1)


In [11]:
print(type(x))

<class 'pandas.core.series.Series'>


In [8]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train,  y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [15]:
avg_length = x.str.split().str.len().mean()
med_len = x.str.split().str.len().median()
max_len = x.str.split().str.len().max()
min_len = x.str.split().str.len().min()
_90th = x.str.split().str.len().quantile(0.9)

In [17]:
print("avg:",avg_length)
print("med",med_len)
print("max",max_len)
print("min",min_len)
print("90 percent:",_90th)

avg: 56.19247683081955
med 26.0
max 12949
min 1
90 percent: 118.0


In [9]:
from sklearn.model_selection import train_test_split
x_small, _, y_small, _ = train_test_split(
    x_train,
    y_train,
    train_size=20000,
    stratify=y_train,
    random_state=42
)

In [10]:
print(x_small.shape,y_small.shape)

(20000,) (20000, 1)


1. increasing data did not work
2. hyperparameter changing didn't worked
Approach:
1. search the different models with their hyperparameters on if-idf with small amt of data with all hyperparameter

Searching all possible models

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
models = [
    LinearSVC,
    RandomForestClassifier,
    SGDClassifier,
    KNeighborsClassifier
]
for model in models:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(min_df = 0.002, max_df = 0.95, ngram_range = (1,2))),
        ("clf", model())
    ])
    scores = cross_validate(
        pipeline,
        x_small,
        y_small,
        cv=5,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
        return_train_score=True,
        n_jobs=-1
    )
    for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        mean = scores[f"test_{metric}"].mean()
        std = scores[f"test_{metric}"].std()
        print(f"model:{model}--------------------------------------------\n")
        print(f"{metric:10}: {mean:.4f} ± {std:.4f}")
        print("\n")

model:<class 'sklearn.svm._classes.LinearSVC'>--------------------------------------------

accuracy  : 0.8598 ± 0.0020


model:<class 'sklearn.svm._classes.LinearSVC'>--------------------------------------------

precision : 0.8785 ± 0.0051


model:<class 'sklearn.svm._classes.LinearSVC'>--------------------------------------------

recall    : 0.8909 ± 0.0035


model:<class 'sklearn.svm._classes.LinearSVC'>--------------------------------------------

f1        : 0.8846 ± 0.0012


model:<class 'sklearn.svm._classes.LinearSVC'>--------------------------------------------

roc_auc   : 0.9332 ± 0.0021


model:<class 'sklearn.ensemble._forest.RandomForestClassifier'>--------------------------------------------

accuracy  : 0.8553 ± 0.0039


model:<class 'sklearn.ensemble._forest.RandomForestClassifier'>--------------------------------------------

precision : 0.8585 ± 0.0048


model:<class 'sklearn.ensemble._forest.RandomForestClassifier'>--------------------------------------------

rec

In [18]:
from xgboost import XGBClassifier
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(min_df = 0.002, max_df = 0.95, ngram_range = (1,2))),
    ("clf", XGBClassifier())
])
scores = cross_validate(
    pipeline,
    x_small,
    y_small,
    cv=5,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    return_train_score=True,
    n_jobs=-1
)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    mean = scores[f"test_{metric}"].mean()
    std = scores[f"test_{metric}"].std()
    print(f"model:{model}--------------------------------------------\n")
    print(f"{metric:10}: {mean:.4f} ± {std:.4f}")
    print("\n")

model:<class 'sklearn.neighbors._classification.KNeighborsClassifier'>--------------------------------------------

accuracy  : 0.8602 ± 0.0038


model:<class 'sklearn.neighbors._classification.KNeighborsClassifier'>--------------------------------------------

precision : 0.8799 ± 0.0024


model:<class 'sklearn.neighbors._classification.KNeighborsClassifier'>--------------------------------------------

recall    : 0.8898 ± 0.0049


model:<class 'sklearn.neighbors._classification.KNeighborsClassifier'>--------------------------------------------

f1        : 0.8848 ± 0.0033


model:<class 'sklearn.neighbors._classification.KNeighborsClassifier'>--------------------------------------------

roc_auc   : 0.9359 ± 0.0026




# Base Line validation accuracy
* accuracy  : 0.7431 ± 0.0054
* precision : 0.7711 ± 0.0028
* recall    : 0.8167 ± 0.0106
* f1        : 0.7932 ± 0.0055
* roc_auc   : 0.8110 ± 0.0044

In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_validate
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(min_df = 0.002, max_df = 0.95, ngram_range = (1,2))),
    ("clf", MultinomialNB())
])
scores = cross_validate(
    pipeline,
    x_small,
    y_small,
    cv=5,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    return_train_score=True,
    n_jobs=-1
)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    mean = scores[f"test_{metric}"].mean()
    std = scores[f"test_{metric}"].std()
    # print(f"model:{model}--------------------------------------------\n")
    print(f"{metric:10}: {mean:.4f} ± {std:.4f}")
    print("\n")

accuracy  : 0.8016 ± 0.0048


precision : 0.8088 ± 0.0025


recall    : 0.8788 ± 0.0070


f1        : 0.8424 ± 0.0043


roc_auc   : 0.8852 ± 0.0046




hyperparameter tuining the sgd, randomforest

In [16]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define the base Pipeline cleanly
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=0.002, max_df=0.95, ngram_range=(1, 2))),
    ("clf", SGDClassifier(loss='hinge', random_state=42))
])

# 2. Define metrics (Use 'f1' as the primary scoring metric for search)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# 3. Prefix parameter grid keys with 'clf__' so the search knows they belong to SGDClassifier
sgd_param_grid = {
    # --- Regularization & Penalty ---
    'clf__penalty': ['l2', 'l1', 'elasticnet'],
    'clf__alpha': [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'clf__l1_ratio': [0.15, 0.30, 0.50, 0.70],

    # --- Learning Rate & Step Size ---
    'clf__learning_rate': ['optimal', 'adaptive', 'invscaling'],
    'clf__eta0': [1e-4, 1e-3, 1e-2, 0.1],
    'clf__power_t': [0.1, 0.25, 0.5],

    # --- Loss & Convergence ---
    'clf__max_iter': [1000, 2000, 3000],
    'clf__tol': [1e-4, 1e-3, 1e-2],
    'clf__early_stopping': [True, False],
    'clf__n_iter_no_change': [5, 10],

    # --- Weighting & Optimization ---
    'clf__class_weight': ['balanced', None],
    'clf__average': [False, True, 10, 50],
    'clf__fit_intercept': [True, False]
}

# 4. Configure RandomizedSearchCV (Random sampling is vastly superior to GridSearch for large grids)
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=sgd_param_grid,
    n_iter=20,                     # Samples 20 random parameter combinations
    scoring=scoring_metrics,
    refit='f1',                    # Refits the best pipeline using F1-score
    cv=5,
    n_jobs=-1,
    verbose=2,                     # Level 2 or 3 gives clean logs without flooding the console
    random_state=42
)

# 5. Execute search on your small subset
print("[Layer 2] Tuning SGD Pipeline...")
random_search.fit(x_small, y_small)

print("\n=== Best Pipeline Score & Parameters ===")
print(f"Best CV F1-Score: {random_search.best_score_:.4f}")
print(random_search.best_params_)

[Layer 2] Tuning SGD Pipeline...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



=== Best Pipeline Score & Parameters ===
Best CV F1-Score: 0.8873
{'clf__tol': 0.0001, 'clf__power_t': 0.1, 'clf__penalty': 'l2', 'clf__n_iter_no_change': 10, 'clf__max_iter': 2000, 'clf__learning_rate': 'adaptive', 'clf__l1_ratio': 0.15, 'clf__fit_intercept': True, 'clf__eta0': 0.001, 'clf__early_stopping': False, 'clf__class_weight': None, 'clf__average': 10, 'clf__alpha': 1e-06}


In [19]:
results_df = pd.DataFrame(random_search.cv_results_)
best_index = random_search.best_index_

print("=== Best Parameter Set Cross-Validation Results ===")
print(f"Mean Accuracy:  {results_df.loc[best_index, 'mean_test_accuracy'] * 100:.2f}%")
print(f"Mean Precision: {results_df.loc[best_index, 'mean_test_precision']:.4f}")
print(f"Mean Recall:    {results_df.loc[best_index, 'mean_test_recall']:.4f}")
print(f"Mean F1-Score:  {results_df.loc[best_index, 'mean_test_f1']:.4f}")
print(f"Mean ROC-AUC:   {results_df.loc[best_index, 'mean_test_roc_auc']:.4f}")

=== Best Parameter Set Cross-Validation Results ===
Mean Accuracy:  86.08%
Mean Precision: 0.8671
Mean Recall:    0.9085
Mean F1-Score:  0.8873
Mean ROC-AUC:   0.9323


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=0.002, max_df=0.95, ngram_range=(1, 2))),
    ("clf", RandomForestClassifier())
])
random_search = RandomForestClassifier(

)

In [ ]:
# model stacking
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# =====================================================================
# LAYER 1: Feature Extraction (Vectorization)
# =====================================================================
tfidf_step = TfidfVectorizer(
    min_df=0.002,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# =====================================================================
# LAYER 2: Level-0 Base Models (Diverse Classifier Pool)
# =====================================================================
# Note: For SGD with hinge loss, set loss='modified_huber' or wrap with
# CalibratedClassifierCV so it outputs smooth probability estimates to Level 1.
level0_base_models = [
    ('sgd_svm', SGDClassifier(
        tol= 0.0001, power_t = 0.1, penalty = 'l2', n_iter_no_change =  10, max_iter = 2000, learning_rate = 'adaptive', l1_ratio = 0.15, fit_intercept = True, eta0 = 0.001, early_stopping = False, class_weight = None, average = 10, alpha = 1e-06
    )),
    ('comp_nb', ComplementNB(alpha=0.1)),
    ('extra_trees', ExtraTreesClassifier(
        n_estimators=100,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
]

# =====================================================================
# LAYER 2: Level-1 Meta-Learner
# =====================================================================
level1_meta_learner = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    random_state=42
)

# =====================================================================
# COMBINED STACKING CLASSIFIER
# =====================================================================
stacking_classifier = StackingClassifier(
    estimators=level0_base_models,
    final_estimator=level1_meta_learner,
    cv=5,                  # Uses 5-fold cross-validation for out-of-fold predictions
    n_jobs=-1,
    passthrough=False      # Pass True if you want Meta-Learner to see raw TF-IDF + predictions
)

# Complete End-to-End Pipeline
stacking_pipeline = Pipeline([
    ('tfidf', tfidf_step),
    ('stacking', stacking_classifier)
])

# =====================================================================
# TRAINING & EVALUATION
# =====================================================================
print("[Layer 2] Training 2-Layer Stacking Architecture...")
scores = cross_validate(
    pipeline,
    x_small,
    y_small,
    cv=5,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    return_train_score=True,
    n_jobs=-1
)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    mean = scores[f"test_{metric}"].mean()
    std = scores[f"test_{metric}"].std()
    # print(f"model:{model}--------------------------------------------\n")
    print(f"{metric:10}: {mean:.4f} ± {std:.4f}")
    print("\n")